In [1]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define Project Folder
FOLDERNAME = 'Colab\ Notebooks/102 Category Flower Dataset'

%cd drive/MyDrive/$FOLDERNAME

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/102 Category Flower Dataset


In [2]:
# Define device
import torch
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
print('Device:', device)

Device: cuda


In [3]:
# Load Existing Dataset
import torchvision.datasets as dset
import torchvision.transforms as T

# Define the transformation pipeline
transform = T.Compose([
    T.RandomResizedCrop(224),  # Resize and crop to 224x224, which is the input size for ResNet
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalization for pre-trained models
])
train_data = dset.Flowers102(root='./train', split='train', download=True, transform=transform)
val_data = dset.Flowers102(root='./val', split='val', download=True, transform=transform)

In [4]:
train_data

Dataset Flowers102
    Number of datapoints: 1020
    Root location: ./train
    split=train
    StandardTransform
Transform: Compose(
               RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
               RandomHorizontalFlip(p=0.5)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [5]:
val_data

Dataset Flowers102
    Number of datapoints: 1020
    Root location: ./val
    split=val
    StandardTransform
Transform: Compose(
               RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
               RandomHorizontalFlip(p=0.5)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [6]:
train_data[2][0].shape

torch.Size([3, 224, 224])

In [7]:
# Check Data Dimension
val_data[2][0].shape

torch.Size([3, 224, 224])

In [4]:
num_train = len(train_data)
num_val = len(val_data)
print('Number of training:', num_train)
print('Number of validation:', num_val)

Number of training: 1020
Number of validation: 1020


In [5]:
# Create Mini-batches
from torch.utils.data import DataLoader
mini_trains = DataLoader(train_data, batch_size=32, shuffle=True)
mini_vals = DataLoader(val_data, batch_size=32, shuffle=True)

In [6]:
import torchvision.models as models
import torch.nn as nn
model = models.densenet121(pretrained=True).cuda()  # Load pre-trained ResNet18

# Replace the last fully connected layer
num_ftrs = model.classifier.in_features
model.fc = nn.Linear(num_ftrs, 102)  # 102 is the number of flower categories

model = model.to(device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# Define loss function & optimizer
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

In [8]:
# Training Procedure
def train(num_epoch, model, mini_trains, mini_vals, device, loss_function, optimizer):
  for epoch in range(num_epoch):
    num_iters = 0
    for x, y in mini_trains:
      model.train()
      x = x.to(device)
      y = y.to(device)
      scores = model(x)
      loss = loss_function(scores, y)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      if num_iters % 30 == 0:
        evaluate_predictor(model, epoch, mini_vals, device)
      num_iters += 1

In [9]:
# Validating Procedure
def evaluate_predictor(model, epoch, mini_vals, device):
  model.eval()
  with torch.no_grad():
    acc_count = 0
    for x, y in mini_vals:
      x = x.to(device)
      y = y.to(device)
      scores=model(x)
      predictions=scores.max(1)[1]
      acc = predictions.eq(y).sum().item()
      acc_count += acc
    print(f'Epoch[{epoch+1}] Acc: {acc_count/num_val}')

In [10]:
# Start training
train(20, model, mini_trains, mini_vals, device, loss_function, optimizer)

Epoch[1] Acc: 0.000980392156862745
Epoch[1] Acc: 0.06568627450980392
Epoch[2] Acc: 0.07647058823529412
Epoch[2] Acc: 0.30980392156862746
Epoch[3] Acc: 0.31666666666666665
Epoch[3] Acc: 0.4931372549019608
Epoch[4] Acc: 0.4970588235294118
Epoch[4] Acc: 0.5901960784313726
Epoch[5] Acc: 0.5794117647058824
Epoch[5] Acc: 0.6647058823529411
Epoch[6] Acc: 0.6568627450980392
Epoch[6] Acc: 0.7137254901960784
Epoch[7] Acc: 0.6941176470588235
Epoch[7] Acc: 0.7549019607843137
Epoch[8] Acc: 0.7235294117647059
Epoch[8] Acc: 0.746078431372549
Epoch[9] Acc: 0.7627450980392156
Epoch[9] Acc: 0.7607843137254902
Epoch[10] Acc: 0.7813725490196078
Epoch[10] Acc: 0.7735294117647059
Epoch[11] Acc: 0.792156862745098
Epoch[11] Acc: 0.7686274509803922
Epoch[12] Acc: 0.7813725490196078
Epoch[12] Acc: 0.8049019607843138
Epoch[13] Acc: 0.8127450980392157
Epoch[13] Acc: 0.8068627450980392
Epoch[14] Acc: 0.8147058823529412
Epoch[14] Acc: 0.8049019607843138
Epoch[15] Acc: 0.7901960784313725
Epoch[15] Acc: 0.81176470588